In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression

# Load team-strength data
team_data = pd.read_csv(
    "../data/nba_team_seasons_1986_2026.csv"
)

# Load historical playoff games
playoff_games = pd.read_csv(
    "../data/nba_playoff_games_1986_2025.csv"
)

# Convert dates
playoff_games["Date"] = pd.to_datetime(
    playoff_games["Date"]
)

print("Data loaded successfully.")
print("Team-seasons:", len(team_data))
print("Playoff games:", len(playoff_games))

Data loaded successfully.
Team-seasons: 1177
Playoff games: 3143


In [2]:
def train_probability_model(target_end_year):
    """
    Train the playoff win-probability model for a target postseason
    using only the previous 7 completed postseasons.
    """

    # Previous 7 postseasons only
    training_games = playoff_games[
        (playoff_games["Season_End"] < target_end_year) &
        (playoff_games["Season_End"] >= target_end_year - 7)
    ].copy()

    # Team SRS data
    srs_data = team_data[
        ["Season", "Team", "SRS"]
    ].copy()

    # Home-team SRS
    home_srs = srs_data.rename(
        columns={
            "Team": "Home_Team",
            "SRS": "Home_SRS"
        }
    )

    # Away-team SRS
    away_srs = srs_data.rename(
        columns={
            "Team": "Away_Team",
            "SRS": "Away_SRS"
        }
    )

    # Merge SRS onto historical playoff games
    training_games = training_games.merge(
        home_srs,
        on=["Season", "Home_Team"],
        how="left"
    )

    training_games = training_games.merge(
        away_srs,
        on=["Season", "Away_Team"],
        how="left"
    )

    # Difference from home team's perspective
    training_games["SRS_Diff"] = (
        training_games["Home_SRS"] -
        training_games["Away_SRS"]
    )

    # Fit logistic regression
    X = training_games[["SRS_Diff"]]
    y = training_games["Home_Win"]

    model = LogisticRegression()
    model.fit(X, y)

    return model

In [3]:
def get_game_probability(
    target_end_year,
    home_team,
    away_team,
    model=None
):
    """
    Return the probability that the home team wins a playoff game.
    """

    season_name = (
        f"{target_end_year - 1}-"
        f"{str(target_end_year)[-2:]}"
    )

    season_teams = team_data[
        team_data["Season"] == season_name
    ]

    home_row = season_teams[
        season_teams["Team"] == home_team
    ]

    away_row = season_teams[
        season_teams["Team"] == away_team
    ]

    if len(home_row) != 1:
        raise ValueError(
            f"Could not uniquely find {home_team} in {season_name}"
        )

    if len(away_row) != 1:
        raise ValueError(
            f"Could not uniquely find {away_team} in {season_name}"
        )

    home_srs = home_row["SRS"].iloc[0]
    away_srs = away_row["SRS"].iloc[0]

    srs_diff = home_srs - away_srs

    # Train model if one wasn't supplied
    if model is None:
        model = train_probability_model(
            target_end_year
        )

    probability = model.predict_proba(
        pd.DataFrame(
            {"SRS_Diff": [srs_diff]}
        )
    )[0, 1]

    return probability

In [4]:
model_2016 = train_probability_model(2016)

gsw_game7_prob = get_game_probability(
    target_end_year=2016,
    home_team="Golden State Warriors",
    away_team="Cleveland Cavaliers",
    model=model_2016
)

print(
    "Golden State home win probability:",
    f"{gsw_game7_prob:.1%}"
)

print(
    "Cleveland away win probability:",
    f"{1 - gsw_game7_prob:.1%}"
)

print("\n2016 model parameters:")
print(
    "Intercept:",
    round(model_2016.intercept_[0], 3)
)
print(
    "SRS coefficient:",
    round(model_2016.coef_[0][0], 3)
)

Golden State home win probability: 79.1%
Cleveland away win probability: 20.9%

2016 model parameters:
Intercept: 0.585
SRS coefficient: 0.151


In [5]:
def simulate_series(
    target_end_year,
    home_court_team,
    opponent,
    model,
    rng=None
):
    """
    Simulate one best-of-seven NBA playoff series.

    home_court_team hosts Games 1, 2, 5, and 7.
    opponent hosts Games 3, 4, and 6.
    """

    if rng is None:
        rng = np.random.default_rng()

    home_court_wins = 0
    opponent_wins = 0

    # 1 = home-court team hosts
    # 0 = opponent hosts
    home_pattern = [1, 1, 0, 0, 1, 0, 1]

    game_results = []

    for game_number, home_indicator in enumerate(
        home_pattern,
        start=1
    ):

        if home_court_wins == 4 or opponent_wins == 4:
            break

        if home_indicator == 1:

            game_home = home_court_team
            game_away = opponent

        else:

            game_home = opponent
            game_away = home_court_team

        home_win_prob = get_game_probability(
            target_end_year=target_end_year,
            home_team=game_home,
            away_team=game_away,
            model=model
        )

        home_team_won = (
            rng.random() < home_win_prob
        )

        if home_team_won:
            winner = game_home
        else:
            winner = game_away

        if winner == home_court_team:
            home_court_wins += 1
        else:
            opponent_wins += 1

        game_results.append({
            "Game": game_number,
            "Home": game_home,
            "Away": game_away,
            "Home_Win_Prob": home_win_prob,
            "Winner": winner
        })

    if home_court_wins == 4:
        series_winner = home_court_team
    else:
        series_winner = opponent

    return {
        "Winner": series_winner,
        "Home_Court_Wins": home_court_wins,
        "Opponent_Wins": opponent_wins,
        "Games": game_results
    }

In [6]:
rng = np.random.default_rng(42)

example_series = simulate_series(
    target_end_year=2016,
    home_court_team="Golden State Warriors",
    opponent="Cleveland Cavaliers",
    model=model_2016,
    rng=rng
)

print("Series winner:", example_series["Winner"])

print(
    "Series score:",
    example_series["Home_Court_Wins"],
    "-",
    example_series["Opponent_Wins"]
)

pd.DataFrame(example_series["Games"])

Series winner: Golden State Warriors
Series score: 4 - 0


,Game,Home,Away,Home_Win_Prob,Winner
0,1,Golden State Warriors,Cleveland Cavaliers,0.790872,Golden State Warriors
1,2,Golden State Warriors,Cleveland Cavaliers,0.790872,Golden State Warriors
2,3,Cleveland Cavaliers,Golden State Warriors,0.460131,Golden State Warriors
3,4,Cleveland Cavaliers,Golden State Warriors,0.460131,Golden State Warriors


In [7]:
# ---------------------------------------
# TEST: SIMULATE 2016 FINALS MATCHUP
# 10,000 TIMES
# ---------------------------------------

n_simulations = 10000

rng = np.random.default_rng(42)

warriors_series_wins = 0
cavaliers_series_wins = 0

series_lengths = []

for _ in range(n_simulations):

    result = simulate_series(
        target_end_year=2016,
        home_court_team="Golden State Warriors",
        opponent="Cleveland Cavaliers",
        model=model_2016,
        rng=rng
    )

    if result["Winner"] == "Golden State Warriors":
        warriors_series_wins += 1
    else:
        cavaliers_series_wins += 1

    series_lengths.append(
        len(result["Games"])
    )

warriors_series_prob = (
    warriors_series_wins / n_simulations
)

cavaliers_series_prob = (
    cavaliers_series_wins / n_simulations
)

print("2016 Warriors vs Cavaliers")
print("--------------------------")

print(
    "Golden State series win probability:",
    f"{warriors_series_prob:.1%}"
)

print(
    "Cleveland series win probability:",
    f"{cavaliers_series_prob:.1%}"
)

print(
    "\nAverage series length:",
    round(np.mean(series_lengths), 2)
)

print("\nSeries length distribution:")

for length in range(4, 8):

    count = series_lengths.count(length)

    print(
        f"{length} games:",
        f"{count / n_simulations:.1%}"
    )

2016 Warriors vs Cavaliers
--------------------------
Golden State series win probability: 86.4%
Cleveland series win probability: 13.6%

Average series length: 5.5

Series length distribution:
4 games: 18.8%
5 games: 34.9%
6 games: 23.1%
7 games: 23.1%


In [8]:
bracket_2016 = {
    "East": {
        "seeds": {
            1: "Cleveland Cavaliers",
            2: "Toronto Raptors",
            3: "Miami Heat",
            4: "Atlanta Hawks",
            5: "Boston Celtics",
            6: "Charlotte Hornets",
            7: "Indiana Pacers",
            8: "Detroit Pistons"
        }
    },

    "West": {
        "seeds": {
            1: "Golden State Warriors",
            2: "San Antonio Spurs",
            3: "Oklahoma City Thunder",
            4: "Los Angeles Clippers",
            5: "Portland Trail Blazers",
            6: "Dallas Mavericks",
            7: "Memphis Grizzlies",
            8: "Houston Rockets"
        }
    }
}

print("2016 playoff bracket loaded.")

2016 playoff bracket loaded.


In [9]:
def get_seed(bracket, conference, team):
    """
    Return a team's playoff seed.
    """

    for seed, team_name in bracket[conference]["seeds"].items():

        if team_name == team:
            return seed

    raise ValueError(
        f"{team} not found in {conference} bracket."
    )

In [10]:
print(
    "Golden State seed:",
    get_seed(
        bracket_2016,
        "West",
        "Golden State Warriors"
    )
)

print(
    "Portland seed:",
    get_seed(
        bracket_2016,
        "West",
        "Portland Trail Blazers"
    )
)

Golden State seed: 1
Portland seed: 5


In [11]:
def simulate_conference(
    target_end_year,
    bracket,
    conference,
    model,
    rng
):
    """
    Simulate one conference playoff bracket:
    First Round -> Semifinals -> Conference Finals
    """

    seeds = bracket[conference]["seeds"]

    # -------------------------
    # FIRST ROUND
    # -------------------------

    first_round_matchups = [
        (1, 8),
        (4, 5),
        (2, 7),
        (3, 6)
    ]

    first_round_winners = {}

    for seed_a, seed_b in first_round_matchups:

        team_a = seeds[seed_a]
        team_b = seeds[seed_b]

        # Lower seed number has home court
        if seed_a < seed_b:
            home_court_team = team_a
            opponent = team_b
        else:
            home_court_team = team_b
            opponent = team_a

        result = simulate_series(
            target_end_year=target_end_year,
            home_court_team=home_court_team,
            opponent=opponent,
            model=model,
            rng=rng
        )

        winner = result["Winner"]

        winner_seed = get_seed(
            bracket,
            conference,
            winner
        )

        first_round_winners[
            (seed_a, seed_b)
        ] = (winner_seed, winner)

    # -------------------------
    # CONFERENCE SEMIFINALS
    # -------------------------

    semifinal_matchups = [
        (
            first_round_winners[(1, 8)],
            first_round_winners[(4, 5)]
        ),
        (
            first_round_winners[(2, 7)],
            first_round_winners[(3, 6)]
        )
    ]

    semifinal_winners = []

    for team_a_info, team_b_info in semifinal_matchups:

        seed_a, team_a = team_a_info
        seed_b, team_b = team_b_info

        if seed_a < seed_b:
            home_court_team = team_a
            opponent = team_b
        else:
            home_court_team = team_b
            opponent = team_a

        result = simulate_series(
            target_end_year=target_end_year,
            home_court_team=home_court_team,
            opponent=opponent,
            model=model,
            rng=rng
        )

        winner = result["Winner"]

        winner_seed = get_seed(
            bracket,
            conference,
            winner
        )

        semifinal_winners.append(
            (winner_seed, winner)
        )

    # -------------------------
    # CONFERENCE FINALS
    # -------------------------

    seed_a, team_a = semifinal_winners[0]
    seed_b, team_b = semifinal_winners[1]

    if seed_a < seed_b:
        home_court_team = team_a
        opponent = team_b
    else:
        home_court_team = team_b
        opponent = team_a

    result = simulate_series(
        target_end_year=target_end_year,
        home_court_team=home_court_team,
        opponent=opponent,
        model=model,
        rng=rng
    )

    conference_champion = result["Winner"]

    conference_champion_seed = get_seed(
        bracket,
        conference,
        conference_champion
    )

    return {
        "Champion": conference_champion,
        "Seed": conference_champion_seed
    }

In [12]:
rng = np.random.default_rng(42)

west_result = simulate_conference(
    target_end_year=2016,
    bracket=bracket_2016,
    conference="West",
    model=model_2016,
    rng=rng
)

print("Simulated Western Conference champion:")
print(
    f"({west_result['Seed']}) "
    f"{west_result['Champion']}"
)

Simulated Western Conference champion:
(2) San Antonio Spurs


In [13]:
def get_regular_season_wins(
    target_end_year,
    team
):
    """
    Return a team's regular-season wins
    for the target season.
    """

    season_name = (
        f"{target_end_year - 1}-"
        f"{str(target_end_year)[-2:]}"
    )

    team_row = team_data[
        (team_data["Season"] == season_name) &
        (team_data["Team"] == team)
    ]

    if len(team_row) != 1:
        raise ValueError(
            f"Could not uniquely find {team} in {season_name}"
        )

    return team_row["W"].iloc[0]

In [14]:
def simulate_postseason(
    target_end_year,
    bracket,
    model,
    rng
):
    """
    Simulate one complete NBA postseason.
    """

    # Simulate Eastern Conference
    east_result = simulate_conference(
        target_end_year=target_end_year,
        bracket=bracket,
        conference="East",
        model=model,
        rng=rng
    )

    # Simulate Western Conference
    west_result = simulate_conference(
        target_end_year=target_end_year,
        bracket=bracket,
        conference="West",
        model=model,
        rng=rng
    )

    east_champion = east_result["Champion"]
    west_champion = west_result["Champion"]

    # Determine Finals home court using regular-season wins
    east_wins = get_regular_season_wins(
        target_end_year,
        east_champion
    )

    west_wins = get_regular_season_wins(
        target_end_year,
        west_champion
    )

    if east_wins > west_wins:
        finals_home_court = east_champion
        finals_opponent = west_champion

    elif west_wins > east_wins:
        finals_home_court = west_champion
        finals_opponent = east_champion

    else:
        # Temporary fallback for exact record ties.
        # We'll handle historical NBA tiebreak rules
        # explicitly before scaling across all seasons.
        east_srs = team_data.loc[
            (team_data["Season_End"] == target_end_year) &
            (team_data["Team"] == east_champion),
            "SRS"
        ].iloc[0]

        west_srs = team_data.loc[
            (team_data["Season_End"] == target_end_year) &
            (team_data["Team"] == west_champion),
            "SRS"
        ].iloc[0]

        if east_srs >= west_srs:
            finals_home_court = east_champion
            finals_opponent = west_champion
        else:
            finals_home_court = west_champion
            finals_opponent = east_champion

    # Simulate NBA Finals
    finals_result = simulate_series(
        target_end_year=target_end_year,
        home_court_team=finals_home_court,
        opponent=finals_opponent,
        model=model,
        rng=rng
    )

    return {
        "East_Champion": east_champion,
        "West_Champion": west_champion,
        "Finals_Home_Court": finals_home_court,
        "Champion": finals_result["Winner"]
    }

In [15]:
rng = np.random.default_rng(42)

alternate_2016 = simulate_postseason(
    target_end_year=2016,
    bracket=bracket_2016,
    model=model_2016,
    rng=rng
)

print("ALTERNATE 2016 NBA PLAYOFFS")
print("---------------------------")

print(
    "Eastern Conference Champion:",
    alternate_2016["East_Champion"]
)

print(
    "Western Conference Champion:",
    alternate_2016["West_Champion"]
)

print(
    "Finals home court:",
    alternate_2016["Finals_Home_Court"]
)

print(
    "\nNBA Champion:",
    alternate_2016["Champion"]
)

ALTERNATE 2016 NBA PLAYOFFS
---------------------------
Eastern Conference Champion: Toronto Raptors
Western Conference Champion: Golden State Warriors
Finals home court: Golden State Warriors

NBA Champion: Golden State Warriors


In [16]:
from collections import Counter

# ---------------------------------------
# REPLAY THE 2016 POSTSEASON 10,000 TIMES
# ---------------------------------------

n_simulations = 10000
rng = np.random.default_rng(42)

champions = Counter()
east_champions = Counter()
west_champions = Counter()

for _ in range(n_simulations):

    result = simulate_postseason(
        target_end_year=2016,
        bracket=bracket_2016,
        model=model_2016,
        rng=rng
    )

    champions[result["Champion"]] += 1
    east_champions[result["East_Champion"]] += 1
    west_champions[result["West_Champion"]] += 1


# Championship probability table
championship_probs_2016 = pd.DataFrame(
    [
        {
            "Team": team,
            "Championships": count,
            "Championship_Probability": count / n_simulations
        }
        for team, count in champions.items()
    ]
).sort_values(
    "Championship_Probability",
    ascending=False
).reset_index(drop=True)

print("2016 CHAMPIONSHIP PROBABILITIES")
print("--------------------------------")

display(
    championship_probs_2016.style.format(
        {
            "Championship_Probability": "{:.1%}"
        }
    )
)

KeyboardInterrupt: 

In [17]:
def build_probability_cache(
    target_end_year,
    bracket,
    model
):
    """
    Precompute home-win probabilities for every possible
    matchup among playoff teams.
    """

    # All playoff teams
    teams = (
        list(bracket["East"]["seeds"].values()) +
        list(bracket["West"]["seeds"].values())
    )

    season_name = (
        f"{target_end_year - 1}-"
        f"{str(target_end_year)[-2:]}"
    )

    # Get SRS once for every playoff team
    season_data = team_data[
        team_data["Season"] == season_name
    ]

    srs_lookup = dict(
        zip(
            season_data["Team"],
            season_data["SRS"]
        )
    )

    probability_cache = {}

    # Calculate every possible home/away matchup once
    for home_team in teams:

        for away_team in teams:

            if home_team == away_team:
                continue

            srs_diff = (
                srs_lookup[home_team] -
                srs_lookup[away_team]
            )

            probability = model.predict_proba(
                pd.DataFrame(
                    {"SRS_Diff": [srs_diff]}
                )
            )[0, 1]

            probability_cache[
                (home_team, away_team)
            ] = probability

    return probability_cache

In [18]:
probability_cache_2016 = build_probability_cache(
    target_end_year=2016,
    bracket=bracket_2016,
    model=model_2016
)

print(
    "Cached matchups:",
    len(probability_cache_2016)
)

print(
    "GSW home vs CLE:",
    f"{probability_cache_2016[
        ('Golden State Warriors', 'Cleveland Cavaliers')
    ]:.1%}"
)

print(
    "CLE home vs GSW:",
    f"{probability_cache_2016[
        ('Cleveland Cavaliers', 'Golden State Warriors')
    ]:.1%}"
)

Cached matchups: 240
GSW home vs CLE: 79.1%
CLE home vs GSW: 46.0%


In [19]:
def simulate_series_fast(
    home_court_team,
    opponent,
    probability_cache,
    rng
):
    """
    Fast best-of-seven simulation using cached probabilities.
    """

    home_court_wins = 0
    opponent_wins = 0

    # 2-2-1-1-1
    home_pattern = [1, 1, 0, 0, 1, 0, 1]

    for home_indicator in home_pattern:

        if home_court_wins == 4 or opponent_wins == 4:
            break

        if home_indicator == 1:
            game_home = home_court_team
            game_away = opponent
        else:
            game_home = opponent
            game_away = home_court_team

        home_win_prob = probability_cache[
            (game_home, game_away)
        ]

        if rng.random() < home_win_prob:
            winner = game_home
        else:
            winner = game_away

        if winner == home_court_team:
            home_court_wins += 1
        else:
            opponent_wins += 1

    if home_court_wins == 4:
        return home_court_team

    return opponent

In [20]:
def simulate_conference_fast(
    bracket,
    conference,
    probability_cache,
    rng
):
    """
    Fast simulation of one conference bracket.
    """

    seeds = bracket[conference]["seeds"]

    # First round
    first_round_matchups = [
        (1, 8),
        (4, 5),
        (2, 7),
        (3, 6)
    ]

    first_round_winners = {}

    for seed_a, seed_b in first_round_matchups:

        team_a = seeds[seed_a]
        team_b = seeds[seed_b]

        # Better seed has home court
        if seed_a < seed_b:
            home_court_team = team_a
            opponent = team_b
        else:
            home_court_team = team_b
            opponent = team_a

        winner = simulate_series_fast(
            home_court_team,
            opponent,
            probability_cache,
            rng
        )

        winner_seed = get_seed(
            bracket,
            conference,
            winner
        )

        first_round_winners[
            (seed_a, seed_b)
        ] = (winner_seed, winner)

    # Conference semifinals
    semifinal_matchups = [
        (
            first_round_winners[(1, 8)],
            first_round_winners[(4, 5)]
        ),
        (
            first_round_winners[(2, 7)],
            first_round_winners[(3, 6)]
        )
    ]

    semifinal_winners = []

    for team_a_info, team_b_info in semifinal_matchups:

        seed_a, team_a = team_a_info
        seed_b, team_b = team_b_info

        if seed_a < seed_b:
            home_court_team = team_a
            opponent = team_b
        else:
            home_court_team = team_b
            opponent = team_a

        winner = simulate_series_fast(
            home_court_team,
            opponent,
            probability_cache,
            rng
        )

        semifinal_winners.append(
            (
                get_seed(
                    bracket,
                    conference,
                    winner
                ),
                winner
            )
        )

    # Conference Finals
    seed_a, team_a = semifinal_winners[0]
    seed_b, team_b = semifinal_winners[1]

    if seed_a < seed_b:
        home_court_team = team_a
        opponent = team_b
    else:
        home_court_team = team_b
        opponent = team_a

    champion = simulate_series_fast(
        home_court_team,
        opponent,
        probability_cache,
        rng
    )

    return {
        "Champion": champion,
        "Seed": get_seed(
            bracket,
            conference,
            champion
        )
    }

In [21]:
def simulate_postseason_fast(
    target_end_year,
    bracket,
    probability_cache,
    rng
):
    """
    Fast simulation of one complete NBA postseason.
    """

    # Simulate both conferences
    east_result = simulate_conference_fast(
        bracket,
        "East",
        probability_cache,
        rng
    )

    west_result = simulate_conference_fast(
        bracket,
        "West",
        probability_cache,
        rng
    )

    east_champion = east_result["Champion"]
    west_champion = west_result["Champion"]

    # Regular-season wins determine Finals home court
    east_wins = get_regular_season_wins(
        target_end_year,
        east_champion
    )

    west_wins = get_regular_season_wins(
        target_end_year,
        west_champion
    )

    if east_wins > west_wins:
        finals_home_court = east_champion
        finals_opponent = west_champion

    elif west_wins > east_wins:
        finals_home_court = west_champion
        finals_opponent = east_champion

    else:
        # Temporary tie fallback
        # We'll replace this before historical scaling
        season_name = (
            f"{target_end_year - 1}-"
            f"{str(target_end_year)[-2:]}"
        )

        season_data = team_data[
            team_data["Season"] == season_name
        ]

        east_srs = season_data.loc[
            season_data["Team"] == east_champion,
            "SRS"
        ].iloc[0]

        west_srs = season_data.loc[
            season_data["Team"] == west_champion,
            "SRS"
        ].iloc[0]

        if east_srs >= west_srs:
            finals_home_court = east_champion
            finals_opponent = west_champion
        else:
            finals_home_court = west_champion
            finals_opponent = east_champion

    # NBA Finals
    champion = simulate_series_fast(
        finals_home_court,
        finals_opponent,
        probability_cache,
        rng
    )

    return {
        "East_Champion": east_champion,
        "West_Champion": west_champion,
        "Champion": champion
    }

In [22]:
import time

rng = np.random.default_rng(42)

start_time = time.perf_counter()

test_champions = Counter()

for _ in range(100):

    result = simulate_postseason_fast(
        target_end_year=2016,
        bracket=bracket_2016,
        probability_cache=probability_cache_2016,
        rng=rng
    )

    test_champions[result["Champion"]] += 1

elapsed = time.perf_counter() - start_time

print("100 simulations complete.")
print(f"Time: {elapsed:.3f} seconds")

print("\nChampions:")
print(test_champions)

100 simulations complete.
Time: 0.089 seconds

Champions:
Counter({'Golden State Warriors': 47, 'San Antonio Spurs': 38, 'Cleveland Cavaliers': 7, 'Atlanta Hawks': 3, 'Oklahoma City Thunder': 3, 'Los Angeles Clippers': 2})


In [23]:
# ---------------------------------------
# 2016 NBA PLAYOFFS — 10,000 REPLAYS
# FAST VERSION
# ---------------------------------------

n_simulations = 10000

rng = np.random.default_rng(42)

champions = Counter()
east_champions = Counter()
west_champions = Counter()

start_time = time.perf_counter()

for _ in range(n_simulations):

    result = simulate_postseason_fast(
        target_end_year=2016,
        bracket=bracket_2016,
        probability_cache=probability_cache_2016,
        rng=rng
    )

    champions[result["Champion"]] += 1
    east_champions[result["East_Champion"]] += 1
    west_champions[result["West_Champion"]] += 1

elapsed = time.perf_counter() - start_time

# Include all 16 playoff teams, even if a team won 0 simulations
all_2016_teams = (
    list(bracket_2016["East"]["seeds"].values()) +
    list(bracket_2016["West"]["seeds"].values())
)

championship_probs_2016 = pd.DataFrame({
    "Team": all_2016_teams
})

championship_probs_2016["Championships"] = (
    championship_probs_2016["Team"]
    .map(champions)
    .fillna(0)
    .astype(int)
)

championship_probs_2016["Championship_Probability"] = (
    championship_probs_2016["Championships"] /
    n_simulations
)

championship_probs_2016 = (
    championship_probs_2016
    .sort_values(
        "Championship_Probability",
        ascending=False
    )
    .reset_index(drop=True)
)

print("2016 NBA PLAYOFFS — 10,000 REPLAYS")
print("----------------------------------")
print(f"Runtime: {elapsed:.2f} seconds")

display(
    championship_probs_2016.style.format({
        "Championship_Probability": "{:.1%}"
    })
)

2016 NBA PLAYOFFS — 10,000 REPLAYS
----------------------------------
Runtime: 5.71 seconds


,Team,Championships,Championship_Probability
0,Golden State Warriors,4827,48.3%
1,San Antonio Spurs,3502,35.0%
2,Cleveland Cavaliers,684,6.8%
3,Oklahoma City Thunder,437,4.4%
4,Toronto Raptors,267,2.7%
5,Atlanta Hawks,112,1.1%
6,Los Angeles Clippers,59,0.6%
7,Charlotte Hornets,39,0.4%
8,Boston Celtics,34,0.3%
9,Miami Heat,23,0.2%


In [24]:
games_2016 = playoff_games[
    playoff_games["Season_End"] == 2016
].copy()

games_2016[
    [
        "Date",
        "Away_Team",
        "Home_Team",
        "Away_PTS",
        "Home_PTS"
    ]
].head(20)

,Date,Away_Team,Home_Team,Away_PTS,Home_PTS
2309,2016-04-16,Indiana Pacers,Toronto Raptors,100,90
2310,2016-04-16,Houston Rockets,Golden State Warriors,78,104
2311,2016-04-16,Boston Celtics,Atlanta Hawks,101,102
2312,2016-04-16,Dallas Mavericks,Oklahoma City Thunder,70,108
2313,2016-04-17,Detroit Pistons,Cleveland Cavaliers,101,106
2314,2016-04-17,Charlotte Hornets,Miami Heat,91,123
2315,2016-04-17,Memphis Grizzlies,San Antonio Spurs,74,106
2316,2016-04-17,Portland Trail Blazers,Los Angeles Clippers,95,115
2317,2016-04-18,Indiana Pacers,Toronto Raptors,87,98
2318,2016-04-18,Dallas Mavericks,Oklahoma City Thunder,85,84


In [25]:
def get_first_round_matchups(target_end_year):
    """
    Identify the unique first-round matchups from the beginning
    of a historical postseason.
    """

    games = playoff_games[
        playoff_games["Season_End"] == target_end_year
    ].sort_values("Date").copy()

    matchups = []
    seen_pairs = set()

    for _, game in games.iterrows():

        team_a = game["Away_Team"]
        team_b = game["Home_Team"]

        # Treat A vs B and B vs A as the same series
        pair = frozenset([team_a, team_b])

        if pair not in seen_pairs:
            seen_pairs.add(pair)

            matchups.append({
                "Team_1": team_a,
                "Team_2": team_b,
                "First_Game_Home": team_b
            })

        # Modern postseason has 8 first-round series
        if len(matchups) == 8:
            break

    return pd.DataFrame(matchups)

In [26]:
first_round_2016 = get_first_round_matchups(2016)

first_round_2016

,Team_1,Team_2,First_Game_Home
0,Indiana Pacers,Toronto Raptors,Toronto Raptors
1,Houston Rockets,Golden State Warriors,Golden State Warriors
2,Boston Celtics,Atlanta Hawks,Atlanta Hawks
3,Dallas Mavericks,Oklahoma City Thunder,Oklahoma City Thunder
4,Detroit Pistons,Cleveland Cavaliers,Cleveland Cavaliers
5,Charlotte Hornets,Miami Heat,Miami Heat
6,Memphis Grizzlies,San Antonio Spurs,San Antonio Spurs
7,Portland Trail Blazers,Los Angeles Clippers,Los Angeles Clippers


In [27]:
first_round_1986 = get_first_round_matchups(1986)

first_round_1986

,Team_1,Team_2,First_Game_Home
0,Chicago Bulls,Boston Celtics,Boston Celtics
1,Detroit Pistons,Atlanta Hawks,Atlanta Hawks
2,San Antonio Spurs,Los Angeles Lakers,Los Angeles Lakers
3,Sacramento Kings,Houston Rockets,Houston Rockets
4,Utah Jazz,Dallas Mavericks,Dallas Mavericks
5,Portland Trail Blazers,Denver Nuggets,Denver Nuggets
6,New Jersey Nets,Milwaukee Bucks,Milwaukee Bucks
7,Washington Bullets,Philadelphia 76ers,Philadelphia 76ers


In [29]:
standings_url_2016 = (
    "https://www.basketball-reference.com/"
    "playoffs/NBA_2016_standings.html"
)

standings_tables_2016 = pd.read_html(
    standings_url_2016,
    flavor="lxml"
)

print(
    "Number of tables:",
    len(standings_tables_2016)
)

for i, table in enumerate(standings_tables_2016):

    print(f"\nTable {i}")
    print("Shape:", table.shape)
    print("Columns:", table.columns.tolist())

    display(table.head(10))

ValueError: No tables found matching regex '.+'

In [30]:
def add_regular_season_records(
    matchups,
    target_end_year
):
    """
    Add regular-season wins/losses to each team's
    first-round matchup information.
    """

    season_name = (
        f"{target_end_year - 1}-"
        f"{str(target_end_year)[-2:]}"
    )

    season_records = team_data[
        team_data["Season"] == season_name
    ][
        ["Team", "W", "L"]
    ].copy()

    result = matchups.copy()

    # Team 1 record
    result = result.merge(
        season_records.rename(
            columns={
                "Team": "Team_1",
                "W": "Team_1_W",
                "L": "Team_1_L"
            }
        ),
        on="Team_1",
        how="left"
    )

    # Team 2 record
    result = result.merge(
        season_records.rename(
            columns={
                "Team": "Team_2",
                "W": "Team_2_W",
                "L": "Team_2_L"
            }
        ),
        on="Team_2",
        how="left"
    )

    return result

In [31]:
first_round_2016_records = add_regular_season_records(
    first_round_2016,
    2016
)

first_round_2016_records

,Team_1,Team_2,First_Game_Home,Team_1_W,Team_1_L,Team_2_W,Team_2_L
0,Indiana Pacers,Toronto Raptors,Toronto Raptors,45.0,37.0,56.0,26.0
1,Houston Rockets,Golden State Warriors,Golden State Warriors,41.0,41.0,73.0,9.0
2,Boston Celtics,Atlanta Hawks,Atlanta Hawks,48.0,34.0,48.0,34.0
3,Dallas Mavericks,Oklahoma City Thunder,Oklahoma City Thunder,42.0,40.0,55.0,27.0
4,Detroit Pistons,Cleveland Cavaliers,Cleveland Cavaliers,44.0,38.0,57.0,25.0
5,Charlotte Hornets,Miami Heat,Miami Heat,48.0,34.0,48.0,34.0
6,Memphis Grizzlies,San Antonio Spurs,San Antonio Spurs,42.0,40.0,67.0,15.0
7,Portland Trail Blazers,Los Angeles Clippers,Los Angeles Clippers,44.0,38.0,53.0,29.0


In [32]:
playoff_teams_2016 = set(
    first_round_2016["Team_1"]
).union(
    set(first_round_2016["Team_2"])
)

team_data[
    (team_data["Season"] == "2015-16") &
    (team_data["Team"].isin(playoff_teams_2016))
][
    ["Rk", "Team", "W", "L", "SRS"]
].sort_values("Rk")

,Rk,Team,W,L,SRS
873,1.0,San Antonio Spurs,67.0,15.0,10.28
856,2.0,Golden State Warriors,73.0,9.0,10.38
867,3.0,Oklahoma City Thunder,55.0,27.0,7.09
852,4.0,Cleveland Cavaliers,57.0,25.0,5.45
874,5.0,Toronto Raptors,56.0,26.0,4.08
859,6.0,Los Angeles Clippers,53.0,29.0,4.13
847,7.0,Atlanta Hawks,48.0,34.0,3.49
848,8.0,Boston Celtics,48.0,34.0,2.84
850,9.0,Charlotte Hornets,48.0,34.0,2.36
862,11.0,Miami Heat,48.0,34.0,1.50


In [33]:
def get_unique_playoff_matchups(target_end_year):
    """
    Return every unique team-vs-team matchup
    in chronological order for a postseason.
    """

    games = playoff_games[
        playoff_games["Season_End"] == target_end_year
    ].sort_values("Date").copy()

    matchups = []
    seen_pairs = set()

    for _, game in games.iterrows():

        team_a = game["Away_Team"]
        team_b = game["Home_Team"]

        pair = frozenset([team_a, team_b])

        if pair not in seen_pairs:

            seen_pairs.add(pair)

            matchups.append({
                "First_Date": game["Date"],
                "Team_1": team_a,
                "Team_2": team_b,
                "First_Game_Home": team_b
            })

    return pd.DataFrame(matchups)


unique_matchups_2016 = get_unique_playoff_matchups(2016)

unique_matchups_2016

,First_Date,Team_1,Team_2,First_Game_Home
0,2016-04-16,Indiana Pacers,Toronto Raptors,Toronto Raptors
1,2016-04-16,Houston Rockets,Golden State Warriors,Golden State Warriors
2,2016-04-16,Boston Celtics,Atlanta Hawks,Atlanta Hawks
3,2016-04-16,Dallas Mavericks,Oklahoma City Thunder,Oklahoma City Thunder
4,2016-04-17,Detroit Pistons,Cleveland Cavaliers,Cleveland Cavaliers
5,2016-04-17,Charlotte Hornets,Miami Heat,Miami Heat
6,2016-04-17,Memphis Grizzlies,San Antonio Spurs,San Antonio Spurs
7,2016-04-17,Portland Trail Blazers,Los Angeles Clippers,Los Angeles Clippers
8,2016-04-30,Oklahoma City Thunder,San Antonio Spurs,San Antonio Spurs
9,2016-05-01,Portland Trail Blazers,Golden State Warriors,Golden State Warriors


In [34]:
unique_matchups_1986 = get_unique_playoff_matchups(1986)

print("Unique series:", len(unique_matchups_1986))

unique_matchups_1986

Unique series: 15


,First_Date,Team_1,Team_2,First_Game_Home
0,1986-04-17,Chicago Bulls,Boston Celtics,Boston Celtics
1,1986-04-17,Detroit Pistons,Atlanta Hawks,Atlanta Hawks
2,1986-04-17,San Antonio Spurs,Los Angeles Lakers,Los Angeles Lakers
3,1986-04-17,Sacramento Kings,Houston Rockets,Houston Rockets
4,1986-04-18,Utah Jazz,Dallas Mavericks,Dallas Mavericks
5,1986-04-18,Portland Trail Blazers,Denver Nuggets,Denver Nuggets
6,1986-04-18,New Jersey Nets,Milwaukee Bucks,Milwaukee Bucks
7,1986-04-18,Washington Bullets,Philadelphia 76ers,Philadelphia 76ers
8,1986-04-26,Denver Nuggets,Houston Rockets,Houston Rockets
9,1986-04-27,Atlanta Hawks,Boston Celtics,Boston Celtics


In [35]:
def count_series_games(target_end_year):
    """
    Count games played in each unique playoff series.
    """

    games = playoff_games[
        playoff_games["Season_End"] == target_end_year
    ].copy()

    series_counts = {}

    for _, game in games.iterrows():

        pair = frozenset([
            game["Away_Team"],
            game["Home_Team"]
        ])

        if pair not in series_counts:
            series_counts[pair] = {
                "Team_1": game["Away_Team"],
                "Team_2": game["Home_Team"],
                "Games": 0
            }

        series_counts[pair]["Games"] += 1

    return pd.DataFrame(
        series_counts.values()
    )


series_games_1986 = count_series_games(1986)

series_games_1986

,Team_1,Team_2,Games
0,Chicago Bulls,Boston Celtics,3
1,Detroit Pistons,Atlanta Hawks,4
2,San Antonio Spurs,Los Angeles Lakers,3
3,Sacramento Kings,Houston Rockets,3
4,Utah Jazz,Dallas Mavericks,4
5,Portland Trail Blazers,Denver Nuggets,4
6,New Jersey Nets,Milwaukee Bucks,3
7,Washington Bullets,Philadelphia 76ers,5
8,Denver Nuggets,Houston Rockets,6
9,Atlanta Hawks,Boston Celtics,5


In [36]:
series_games_2016 = count_series_games(2016)

series_games_2016

,Team_1,Team_2,Games
0,Indiana Pacers,Toronto Raptors,7
1,Houston Rockets,Golden State Warriors,5
2,Boston Celtics,Atlanta Hawks,6
3,Dallas Mavericks,Oklahoma City Thunder,5
4,Detroit Pistons,Cleveland Cavaliers,4
5,Charlotte Hornets,Miami Heat,7
6,Memphis Grizzlies,San Antonio Spurs,4
7,Portland Trail Blazers,Los Angeles Clippers,6
8,Oklahoma City Thunder,San Antonio Spurs,6
9,Portland Trail Blazers,Golden State Warriors,5


In [37]:
def get_series_winner(
    target_end_year,
    team_a,
    team_b
):
    """
    Determine the actual historical winner of a playoff series.
    Used only to reconstruct bracket topology.
    """

    games = playoff_games[
        (playoff_games["Season_End"] == target_end_year) &
        (
            (
                (playoff_games["Home_Team"] == team_a) &
                (playoff_games["Away_Team"] == team_b)
            )
            |
            (
                (playoff_games["Home_Team"] == team_b) &
                (playoff_games["Away_Team"] == team_a)
            )
        )
    ]

    wins = {
        team_a: 0,
        team_b: 0
    }

    for _, game in games.iterrows():

        if game["Home_Win"] == 1:
            wins[game["Home_Team"]] += 1
        else:
            wins[game["Away_Team"]] += 1

    return max(
        wins,
        key=wins.get
    )

In [38]:
print(
    "2016 GSW-HOU winner:",
    get_series_winner(
        2016,
        "Golden State Warriors",
        "Houston Rockets"
    )
)

print(
    "1986 BOS-CHI winner:",
    get_series_winner(
        1986,
        "Boston Celtics",
        "Chicago Bulls"
    )
)

2016 GSW-HOU winner: Golden State Warriors
1986 BOS-CHI winner: Boston Celtics


In [39]:
def build_round2_branches(target_end_year):
    """
    Determine which two first-round series feed into
    each conference-semifinal series.

    Historical winners are used ONLY to identify
    bracket topology.
    """

    unique_series = get_unique_playoff_matchups(
        target_end_year
    )

    # First 8 unique series = first round
    round1 = unique_series.iloc[:8].copy()

    # Next 4 = conference semifinals
    round2 = unique_series.iloc[8:12].copy()

    # Find the actual winner of each first-round series
    round1_info = []

    for idx, row in round1.iterrows():

        winner = get_series_winner(
            target_end_year,
            row["Team_1"],
            row["Team_2"]
        )

        round1_info.append({
            "Round1_Index": idx,
            "Team_1": row["Team_1"],
            "Team_2": row["Team_2"],
            "Historical_Winner": winner
        })

    round1_info = pd.DataFrame(round1_info)

    branches = []

    # For each historical Round 2 series,
    # find which two Round 1 winners fed into it
    for r2_index, row in round2.iterrows():

        round2_teams = {
            row["Team_1"],
            row["Team_2"]
        }

        feeding_series = round1_info[
            round1_info["Historical_Winner"].isin(
                round2_teams
            )
        ]

        if len(feeding_series) != 2:
            raise ValueError(
                f"Could not reconstruct Round 2 branch "
                f"for {target_end_year}: "
                f"{row['Team_1']} vs {row['Team_2']}"
            )

        branches.append({
            "Round2_Index": r2_index,
            "Historical_Round2_Matchup":
                f"{row['Team_1']} vs {row['Team_2']}",
            "Round1_Series_A":
                int(feeding_series.iloc[0]["Round1_Index"]),
            "Round1_Series_B":
                int(feeding_series.iloc[1]["Round1_Index"])
        })

    return pd.DataFrame(branches)

In [40]:
round2_branches_2016 = build_round2_branches(2016)

round2_branches_2016

,Round2_Index,Historical_Round2_Matchup,Round1_Series_A,Round1_Series_B
0,8,Oklahoma City Thunder vs San Antonio Spurs,3,6
1,9,Portland Trail Blazers vs Golden State Warriors,1,7
2,10,Atlanta Hawks vs Cleveland Cavaliers,2,4
3,11,Miami Heat vs Toronto Raptors,0,5


In [41]:
round2_branches_1986 = build_round2_branches(1986)

round2_branches_1986

,Round2_Index,Historical_Round2_Matchup,Round1_Series_A,Round1_Series_B
0,8,Denver Nuggets vs Houston Rockets,3,5
1,9,Atlanta Hawks vs Boston Celtics,0,1
2,10,Dallas Mavericks vs Los Angeles Lakers,2,4
3,11,Philadelphia 76ers vs Milwaukee Bucks,6,7


In [42]:
def build_round3_branches(target_end_year):
    """
    Determine which two Round 2 series feed into
    each Conference Finals series.

    Historical winners are used only to reconstruct
    bracket topology.
    """

    unique_series = get_unique_playoff_matchups(
        target_end_year
    )

    # Rows 8-11 = Round 2
    round2 = unique_series.iloc[8:12].copy()

    # Rows 12-13 = Conference Finals
    round3 = unique_series.iloc[12:14].copy()

    # Find historical winner of each Round 2 series
    round2_info = []

    for idx, row in round2.iterrows():

        winner = get_series_winner(
            target_end_year,
            row["Team_1"],
            row["Team_2"]
        )

        round2_info.append({
            "Round2_Index": idx,
            "Team_1": row["Team_1"],
            "Team_2": row["Team_2"],
            "Historical_Winner": winner
        })

    round2_info = pd.DataFrame(round2_info)

    branches = []

    # Determine which two Round 2 series
    # feed into each Conference Final
    for r3_index, row in round3.iterrows():

        round3_teams = {
            row["Team_1"],
            row["Team_2"]
        }

        feeding_series = round2_info[
            round2_info["Historical_Winner"].isin(
                round3_teams
            )
        ]

        if len(feeding_series) != 2:
            raise ValueError(
                f"Could not reconstruct Conference Final "
                f"for {target_end_year}: "
                f"{row['Team_1']} vs {row['Team_2']}"
            )

        branches.append({
            "Round3_Index": r3_index,
            "Historical_Conference_Final":
                f"{row['Team_1']} vs {row['Team_2']}",
            "Round2_Series_A":
                int(feeding_series.iloc[0]["Round2_Index"]),
            "Round2_Series_B":
                int(feeding_series.iloc[1]["Round2_Index"])
        })

    return pd.DataFrame(branches)

In [43]:
round3_branches_2016 = build_round3_branches(2016)
round3_branches_2016

,Round3_Index,Historical_Conference_Final,Round2_Series_A,Round2_Series_B
0,12,Oklahoma City Thunder vs Golden State Warriors,8,9
1,13,Toronto Raptors vs Cleveland Cavaliers,10,11


In [44]:
round3_branches_1986 = build_round3_branches(1986)
round3_branches_1986

,Round3_Index,Historical_Conference_Final,Round2_Series_A,Round2_Series_B
0,12,Houston Rockets vs Los Angeles Lakers,8,10
1,13,Milwaukee Bucks vs Boston Celtics,9,11


In [45]:
def build_finals_branch(target_end_year):
    """
    Determine which two Conference Finals feed into
    the NBA Finals.

    Historical winners are used only to reconstruct
    bracket topology.
    """

    unique_series = get_unique_playoff_matchups(
        target_end_year
    )

    # Rows 12-13 = Conference Finals
    round3 = unique_series.iloc[12:14].copy()

    # Row 14 = NBA Finals
    finals = unique_series.iloc[14]

    round3_info = []

    for idx, row in round3.iterrows():

        winner = get_series_winner(
            target_end_year,
            row["Team_1"],
            row["Team_2"]
        )

        round3_info.append({
            "Round3_Index": idx,
            "Historical_Winner": winner
        })

    round3_info = pd.DataFrame(round3_info)

    finals_teams = {
        finals["Team_1"],
        finals["Team_2"]
    }

    feeding_series = round3_info[
        round3_info["Historical_Winner"].isin(
            finals_teams
        )
    ]

    if len(feeding_series) != 2:
        raise ValueError(
            f"Could not reconstruct NBA Finals "
            f"for {target_end_year}"
        )

    return {
        "Finals_Index": 14,
        "Historical_Finals":
            f"{finals['Team_1']} vs {finals['Team_2']}",
        "Round3_Series_A":
            int(feeding_series.iloc[0]["Round3_Index"]),
        "Round3_Series_B":
            int(feeding_series.iloc[1]["Round3_Index"])
    }

In [46]:
finals_branch_2016 = build_finals_branch(2016)
finals_branch_2016

{'Finals_Index': 14,
 'Historical_Finals': 'Cleveland Cavaliers vs Golden State Warriors',
 'Round3_Series_A': 12,
 'Round3_Series_B': 13}

In [47]:
finals_branch_1986 = build_finals_branch(1986)
finals_branch_1986

{'Finals_Index': 14,
 'Historical_Finals': 'Houston Rockets vs Boston Celtics',
 'Round3_Series_A': 12,
 'Round3_Series_B': 13}

In [48]:
def build_historical_bracket(target_end_year):
    """
    Reconstruct the complete playoff bracket topology
    for one historical NBA postseason.
    """

    unique_series = get_unique_playoff_matchups(
        target_end_year
    )

    if len(unique_series) != 15:
        raise ValueError(
            f"{target_end_year} has "
            f"{len(unique_series)} unique series, expected 15."
        )

    # Initial first-round series
    round1 = []

    for idx, row in unique_series.iloc[:8].iterrows():

        round1.append({
            "Series_Index": int(idx),
            "Team_1": row["Team_1"],
            "Team_2": row["Team_2"],
            "Historical_Game1_Home":
                row["First_Game_Home"]
        })

    # Branch connections
    round2_df = build_round2_branches(
        target_end_year
    )

    round3_df = build_round3_branches(
        target_end_year
    )

    finals_info = build_finals_branch(
        target_end_year
    )

    return {
        "Season_End": target_end_year,
        "Round1": round1,
        "Round2": round2_df.to_dict("records"),
        "Round3": round3_df.to_dict("records"),
        "Finals": finals_info
    }

In [49]:
historical_bracket_2016 = build_historical_bracket(2016)

historical_bracket_2016

{'Season_End': 2016,
 'Round1': [{'Series_Index': 0,
   'Team_1': 'Indiana Pacers',
   'Team_2': 'Toronto Raptors',
   'Historical_Game1_Home': 'Toronto Raptors'},
  {'Series_Index': 1,
   'Team_1': 'Houston Rockets',
   'Team_2': 'Golden State Warriors',
   'Historical_Game1_Home': 'Golden State Warriors'},
  {'Series_Index': 2,
   'Team_1': 'Boston Celtics',
   'Team_2': 'Atlanta Hawks',
   'Historical_Game1_Home': 'Atlanta Hawks'},
  {'Series_Index': 3,
   'Team_1': 'Dallas Mavericks',
   'Team_2': 'Oklahoma City Thunder',
   'Historical_Game1_Home': 'Oklahoma City Thunder'},
  {'Series_Index': 4,
   'Team_1': 'Detroit Pistons',
   'Team_2': 'Cleveland Cavaliers',
   'Historical_Game1_Home': 'Cleveland Cavaliers'},
  {'Series_Index': 5,
   'Team_1': 'Charlotte Hornets',
   'Team_2': 'Miami Heat',
   'Historical_Game1_Home': 'Miami Heat'},
  {'Series_Index': 6,
   'Team_1': 'Memphis Grizzlies',
   'Team_2': 'San Antonio Spurs',
   'Historical_Game1_Home': 'San Antonio Spurs'},
  {'Se

In [50]:
successful_brackets = []
failed_brackets = []

for year in range(1986, 2026):

    try:

        bracket = build_historical_bracket(year)

        successful_brackets.append(year)

    except Exception as e:

        failed_brackets.append(
            (year, str(e))
        )

print(
    "Successful seasons:",
    len(successful_brackets)
)

print(
    "Failed seasons:",
    len(failed_brackets)
)

if failed_brackets:

    print("\nFailures:")

    for year, error in failed_brackets:
        print(year, "->", error)

Successful seasons: 40
Failed seasons: 0


In [51]:
def get_round_format(target_end_year, round_number):
    """
    Return the wins needed and home-court pattern
    for a given historical playoff round.

    round_number:
        1 = First Round
        2 = Conference Semifinals
        3 = Conference Finals
        4 = NBA Finals
    """

    # First Round was best-of-five through 2002
    if round_number == 1 and target_end_year <= 2002:

        return {
            "Best_Of": 5,
            "Wins_Needed": 3,

            # 2-2-1 format:
            # better home-court team hosts 1, 2, 5
            "Home_Pattern": [1, 1, 0, 0, 1]
        }

    # Everything else in our period is best-of-seven
    return {
        "Best_Of": 7,
        "Wins_Needed": 4,

        # 2-2-1-1-1
        "Home_Pattern": [1, 1, 0, 0, 1, 0, 1]
    }

In [52]:
print("1986 Round 1:")
print(get_round_format(1986, 1))

print("\n1986 Round 2:")
print(get_round_format(1986, 2))

print("\n2002 Round 1:")
print(get_round_format(2002, 1))

print("\n2003 Round 1:")
print(get_round_format(2003, 1))

print("\n2016 Round 1:")
print(get_round_format(2016, 1))

1986 Round 1:
{'Best_Of': 5, 'Wins_Needed': 3, 'Home_Pattern': [1, 1, 0, 0, 1]}

1986 Round 2:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 1, 0, 1]}

2002 Round 1:
{'Best_Of': 5, 'Wins_Needed': 3, 'Home_Pattern': [1, 1, 0, 0, 1]}

2003 Round 1:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 1, 0, 1]}

2016 Round 1:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 1, 0, 1]}


In [53]:
def get_round_format(target_end_year, round_number):
    """
    Return the historical series format for a playoff round.

    round_number:
        1 = First Round
        2 = Conference Semifinals
        3 = Conference Finals
        4 = NBA Finals
    """

    # -----------------------------------
    # FIRST ROUND
    # Best-of-five through 2002
    # -----------------------------------

    if round_number == 1 and target_end_year <= 2002:

        return {
            "Best_Of": 5,
            "Wins_Needed": 3,

            # 2-2-1
            "Home_Pattern": [1, 1, 0, 0, 1]
        }

    # -----------------------------------
    # NBA FINALS
    # 2-3-2 format through 2013
    # -----------------------------------

    if round_number == 4 and target_end_year <= 2013:

        return {
            "Best_Of": 7,
            "Wins_Needed": 4,

            # Home-court team hosts:
            # Games 1, 2, 6, 7
            "Home_Pattern": [1, 1, 0, 0, 0, 1, 1]
        }

    # -----------------------------------
    # STANDARD BEST-OF-SEVEN
    # 2-2-1-1-1
    # -----------------------------------

    return {
        "Best_Of": 7,
        "Wins_Needed": 4,

        # Home-court team hosts:
        # Games 1, 2, 5, 7
        "Home_Pattern": [1, 1, 0, 0, 1, 0, 1]
    }

In [54]:
print("1986 Round 1:")
print(get_round_format(1986, 1))

print("\n1986 Finals:")
print(get_round_format(1986, 4))

print("\n2013 Finals:")
print(get_round_format(2013, 4))

print("\n2014 Finals:")
print(get_round_format(2014, 4))

print("\n2016 Finals:")
print(get_round_format(2016, 4))

1986 Round 1:
{'Best_Of': 5, 'Wins_Needed': 3, 'Home_Pattern': [1, 1, 0, 0, 1]}

1986 Finals:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 0, 1, 1]}

2013 Finals:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 0, 1, 1]}

2014 Finals:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 1, 0, 1]}

2016 Finals:
{'Best_Of': 7, 'Wins_Needed': 4, 'Home_Pattern': [1, 1, 0, 0, 1, 0, 1]}


In [55]:
def simulate_historical_series_fast(
    target_end_year,
    round_number,
    home_court_team,
    opponent,
    probability_cache,
    rng
):
    """
    Simulate one playoff series using the historically
    correct series length and home-court pattern.
    """

    series_format = get_round_format(
        target_end_year,
        round_number
    )

    wins_needed = series_format["Wins_Needed"]
    home_pattern = series_format["Home_Pattern"]

    home_court_wins = 0
    opponent_wins = 0

    for home_indicator in home_pattern:

        # Stop once the series has been won
        if (
            home_court_wins == wins_needed or
            opponent_wins == wins_needed
        ):
            break

        if home_indicator == 1:
            game_home = home_court_team
            game_away = opponent

        else:
            game_home = opponent
            game_away = home_court_team

        home_win_prob = probability_cache[
            (game_home, game_away)
        ]

        if rng.random() < home_win_prob:
            winner = game_home
        else:
            winner = game_away

        if winner == home_court_team:
            home_court_wins += 1
        else:
            opponent_wins += 1

    if home_court_wins == wins_needed:
        return home_court_team

    return opponent

In [56]:
model_1986 = train_probability_model(1986)

# We'll build a temporary 1986 cache using the 16 playoff teams
historical_bracket_1986 = build_historical_bracket(1986)

playoff_teams_1986 = set()

for series in historical_bracket_1986["Round1"]:
    playoff_teams_1986.add(series["Team_1"])
    playoff_teams_1986.add(series["Team_2"])

season_data_1986 = team_data[
    team_data["Season_End"] == 1986
]

srs_lookup_1986 = dict(
    zip(
        season_data_1986["Team"],
        season_data_1986["SRS"]
    )
)

probability_cache_1986 = {}

for home_team in playoff_teams_1986:
    for away_team in playoff_teams_1986:

        if home_team == away_team:
            continue

        srs_diff = (
            srs_lookup_1986[home_team] -
            srs_lookup_1986[away_team]
        )

        probability_cache_1986[
            (home_team, away_team)
        ] = model_1986.predict_proba(
            pd.DataFrame({
                "SRS_Diff": [srs_diff]
            })
        )[0, 1]

ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by LogisticRegression.

In [57]:
model_1993 = train_probability_model(1993)

print("1993 model trained successfully.")
print(
    "Intercept:",
    round(model_1993.intercept_[0], 3)
)
print(
    "SRS coefficient:",
    round(model_1993.coef_[0][0], 3)
)

1993 model trained successfully.
Intercept: 0.77
SRS coefficient: 0.163


In [58]:
def build_historical_probability_cache(
    target_end_year,
    historical_bracket,
    model
):
    """
    Build cached game probabilities for every possible
    matchup among that season's 16 playoff teams.
    """

    # Collect the 16 first-round playoff teams
    playoff_teams = set()

    for series in historical_bracket["Round1"]:

        playoff_teams.add(series["Team_1"])
        playoff_teams.add(series["Team_2"])

    playoff_teams = list(playoff_teams)

    # Get that season's SRS values
    season_data = team_data[
        team_data["Season_End"] == target_end_year
    ].copy()

    srs_lookup = dict(
        zip(
            season_data["Team"],
            season_data["SRS"]
        )
    )

    probability_cache = {}

    # Calculate each possible matchup once
    for home_team in playoff_teams:

        for away_team in playoff_teams:

            if home_team == away_team:
                continue

            srs_diff = (
                srs_lookup[home_team] -
                srs_lookup[away_team]
            )

            probability_cache[
                (home_team, away_team)
            ] = model.predict_proba(
                pd.DataFrame({
                    "SRS_Diff": [srs_diff]
                })
            )[0, 1]

    return probability_cache

In [59]:
historical_bracket_1993 = build_historical_bracket(
    1993
)

probability_cache_1993 = (
    build_historical_probability_cache(
        target_end_year=1993,
        historical_bracket=historical_bracket_1993,
        model=model_1993
    )
)

print(
    "1993 cached matchups:",
    len(probability_cache_1993)
)

1993 cached matchups: 240


In [60]:
pd.DataFrame(
    historical_bracket_1993["Round1"]
)

,Series_Index,Team_1,Team_2,Historical_Game1_Home
0,0,New Jersey Nets,Cleveland Cavaliers,Cleveland Cavaliers
1,1,Charlotte Hornets,Boston Celtics,Boston Celtics
2,2,San Antonio Spurs,Portland Trail Blazers,Portland Trail Blazers
3,3,Los Angeles Clippers,Houston Rockets,Houston Rockets
4,4,Atlanta Hawks,Chicago Bulls,Chicago Bulls
5,5,Utah Jazz,Seattle SuperSonics,Seattle SuperSonics
6,6,Los Angeles Lakers,Phoenix Suns,Phoenix Suns
7,7,Indiana Pacers,New York Knicks,New York Knicks


In [61]:
rng = np.random.default_rng(42)

winner_1993_test = simulate_historical_series_fast(
    target_end_year=1993,
    round_number=1,
    home_court_team="Cleveland Cavaliers",
    opponent="New Jersey Nets",
    probability_cache=probability_cache_1993,
    rng=rng
)

print(
    "Simulated 1993 CLE-NJN winner:",
    winner_1993_test
)

print(
    "Series format:",
    get_round_format(1993, 1)
)

Simulated 1993 CLE-NJN winner: Cleveland Cavaliers
Series format: {'Best_Of': 5, 'Wins_Needed': 3, 'Home_Pattern': [1, 1, 0, 0, 1]}


In [62]:
def determine_home_court(
    target_end_year,
    team_a,
    team_b
):
    """
    Determine which team receives home-court advantage.

    Uses regular-season wins first.
    Uses SRS as a temporary tiebreak for equal records.
    """

    season_data = team_data[
        team_data["Season_End"] == target_end_year
    ]

    row_a = season_data[
        season_data["Team"] == team_a
    ].iloc[0]

    row_b = season_data[
        season_data["Team"] == team_b
    ].iloc[0]

    # Better regular-season record
    if row_a["W"] > row_b["W"]:
        return team_a, team_b

    if row_b["W"] > row_a["W"]:
        return team_b, team_a

    # Temporary equal-record tiebreak
    if row_a["SRS"] >= row_b["SRS"]:
        return team_a, team_b

    return team_b, team_a

In [63]:
def simulate_historical_postseason(
    target_end_year,
    historical_bracket,
    probability_cache,
    rng
):
    """
    Simulate one complete historical NBA postseason
    using reconstructed bracket topology and the
    historically correct series format.
    """

    # -----------------------------------
    # ROUND 1
    # -----------------------------------

    round1_winners = {}

    for series in historical_bracket["Round1"]:

        series_index = series["Series_Index"]

        team_a = series["Team_1"]
        team_b = series["Team_2"]

        home_court_team, opponent = determine_home_court(
            target_end_year,
            team_a,
            team_b
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=1,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round1_winners[series_index] = winner

    # -----------------------------------
    # ROUND 2
    # -----------------------------------

    round2_winners = {}

    for branch in historical_bracket["Round2"]:

        round2_index = branch["Round2_Index"]

        team_a = round1_winners[
            branch["Round1_Series_A"]
        ]

        team_b = round1_winners[
            branch["Round1_Series_B"]
        ]

        home_court_team, opponent = determine_home_court(
            target_end_year,
            team_a,
            team_b
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=2,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round2_winners[round2_index] = winner

    # -----------------------------------
    # CONFERENCE FINALS
    # -----------------------------------

    round3_winners = {}

    for branch in historical_bracket["Round3"]:

        round3_index = branch["Round3_Index"]

        team_a = round2_winners[
            branch["Round2_Series_A"]
        ]

        team_b = round2_winners[
            branch["Round2_Series_B"]
        ]

        home_court_team, opponent = determine_home_court(
            target_end_year,
            team_a,
            team_b
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=3,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round3_winners[round3_index] = winner

    # -----------------------------------
    # NBA FINALS
    # -----------------------------------

    finals_info = historical_bracket["Finals"]

    team_a = round3_winners[
        finals_info["Round3_Series_A"]
    ]

    team_b = round3_winners[
        finals_info["Round3_Series_B"]
    ]

    home_court_team, opponent = determine_home_court(
        target_end_year,
        team_a,
        team_b
    )

    champion = simulate_historical_series_fast(
        target_end_year=target_end_year,
        round_number=4,
        home_court_team=home_court_team,
        opponent=opponent,
        probability_cache=probability_cache,
        rng=rng
    )

    return champion

In [64]:
rng = np.random.default_rng(42)

champion_1993_test = simulate_historical_postseason(
    target_end_year=1993,
    historical_bracket=historical_bracket_1993,
    probability_cache=probability_cache_1993,
    rng=rng
)

print(
    "Simulated 1993 NBA Champion:",
    champion_1993_test
)

Simulated 1993 NBA Champion: Phoenix Suns


In [65]:
historical_bracket_2016 = build_historical_bracket(
    2016
)

historical_probability_cache_2016 = (
    build_historical_probability_cache(
        target_end_year=2016,
        historical_bracket=historical_bracket_2016,
        model=model_2016
    )
)

print(
    "2016 cached matchups:",
    len(historical_probability_cache_2016)
)

2016 cached matchups: 240


In [66]:
# ---------------------------------------
# VALIDATE GENERALIZED ENGINE ON 2016
# ---------------------------------------

n_simulations = 10000

rng = np.random.default_rng(42)

generalized_2016_champions = Counter()

start_time = time.perf_counter()

for _ in range(n_simulations):

    champion = simulate_historical_postseason(
        target_end_year=2016,
        historical_bracket=historical_bracket_2016,
        probability_cache=historical_probability_cache_2016,
        rng=rng
    )

    generalized_2016_champions[champion] += 1

elapsed = time.perf_counter() - start_time

generalized_2016_results = pd.DataFrame(
    [
        {
            "Team": team,
            "Championships": count,
            "Championship_Probability":
                count / n_simulations
        }
        for team, count
        in generalized_2016_champions.items()
    ]
).sort_values(
    "Championship_Probability",
    ascending=False
).reset_index(drop=True)

print(
    f"Runtime: {elapsed:.2f} seconds"
)

display(
    generalized_2016_results.style.format({
        "Championship_Probability": "{:.1%}"
    })
)

Runtime: 77.32 seconds


,Team,Championships,Championship_Probability
0,Golden State Warriors,4859,48.6%
1,San Antonio Spurs,3449,34.5%
2,Cleveland Cavaliers,730,7.3%
3,Oklahoma City Thunder,445,4.5%
4,Toronto Raptors,237,2.4%
5,Atlanta Hawks,108,1.1%
6,Los Angeles Clippers,49,0.5%
7,Charlotte Hornets,46,0.5%
8,Boston Celtics,36,0.4%
9,Miami Heat,21,0.2%


In [67]:
def build_team_info_cache(target_end_year):
    """
    Cache regular-season wins and SRS for every team
    so simulations require no pandas lookups.
    """

    season_data = team_data[
        team_data["Season_End"] == target_end_year
    ]

    return {
        row["Team"]: {
            "W": row["W"],
            "SRS": row["SRS"]
        }
        for _, row in season_data.iterrows()
    }


def determine_home_court_fast(
    team_a,
    team_b,
    team_info_cache
):
    """
    Fast home-court determination using cached data.
    """

    info_a = team_info_cache[team_a]
    info_b = team_info_cache[team_b]

    if info_a["W"] > info_b["W"]:
        return team_a, team_b

    if info_b["W"] > info_a["W"]:
        return team_b, team_a

    # Temporary equal-record tiebreak
    if info_a["SRS"] >= info_b["SRS"]:
        return team_a, team_b

    return team_b, team_a

In [68]:
def simulate_historical_postseason_fast(
    target_end_year,
    historical_bracket,
    probability_cache,
    team_info_cache,
    rng
):
    """
    Fully optimized historical postseason simulation.
    No pandas operations occur inside the simulation.
    """

    # -----------------------------
    # ROUND 1
    # -----------------------------

    round1_winners = {}

    for series in historical_bracket["Round1"]:

        series_index = series["Series_Index"]

        team_a = series["Team_1"]
        team_b = series["Team_2"]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=1,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round1_winners[series_index] = winner

    # -----------------------------
    # ROUND 2
    # -----------------------------

    round2_winners = {}

    for branch in historical_bracket["Round2"]:

        round2_index = branch["Round2_Index"]

        team_a = round1_winners[
            branch["Round1_Series_A"]
        ]

        team_b = round1_winners[
            branch["Round1_Series_B"]
        ]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=2,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round2_winners[round2_index] = winner

    # -----------------------------
    # CONFERENCE FINALS
    # -----------------------------

    round3_winners = {}

    for branch in historical_bracket["Round3"]:

        round3_index = branch["Round3_Index"]

        team_a = round2_winners[
            branch["Round2_Series_A"]
        ]

        team_b = round2_winners[
            branch["Round2_Series_B"]
        ]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=3,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round3_winners[round3_index] = winner

    # -----------------------------
    # NBA FINALS
    # -----------------------------

    finals_info = historical_bracket["Finals"]

    team_a = round3_winners[
        finals_info["Round3_Series_A"]
    ]

    team_b = round3_winners[
        finals_info["Round3_Series_B"]
    ]

    home_court_team, opponent = (
        determine_home_court_fast(
            team_a,
            team_b,
            team_info_cache
        )
    )

    champion = simulate_historical_series_fast(
        target_end_year=target_end_year,
        round_number=4,
        home_court_team=home_court_team,
        opponent=opponent,
        probability_cache=probability_cache,
        rng=rng
    )

    return champion

In [69]:
team_info_cache_2016 = build_team_info_cache(2016)

In [70]:
rng = np.random.default_rng(42)

start_time = time.perf_counter()

test_champions_fast = Counter()

for _ in range(100):

    champion = simulate_historical_postseason_fast(
        target_end_year=2016,
        historical_bracket=historical_bracket_2016,
        probability_cache=historical_probability_cache_2016,
        team_info_cache=team_info_cache_2016,
        rng=rng
    )

    test_champions_fast[champion] += 1

elapsed = time.perf_counter() - start_time

print("100 optimized historical simulations complete.")
print(f"Time: {elapsed:.3f} seconds")
print(test_champions_fast)

100 optimized historical simulations complete.
Time: 0.010 seconds
Counter({'Golden State Warriors': 53, 'San Antonio Spurs': 29, 'Cleveland Cavaliers': 7, 'Charlotte Hornets': 4, 'Oklahoma City Thunder': 3, 'Los Angeles Clippers': 2, 'Atlanta Hawks': 1, 'Boston Celtics': 1})


In [71]:
# ---------------------------------------
# HOW COMMON ARE EQUAL-RECORD MATCHUPS?
# ---------------------------------------

equal_record_summary = []

for year in range(1993, 2026):

    bracket = build_historical_bracket(year)

    playoff_teams = set()

    for series in bracket["Round1"]:
        playoff_teams.add(series["Team_1"])
        playoff_teams.add(series["Team_2"])

    info = build_team_info_cache(year)

    teams = sorted(playoff_teams)

    equal_pairs = []

    for i in range(len(teams)):
        for j in range(i + 1, len(teams)):

            team_a = teams[i]
            team_b = teams[j]

            if info[team_a]["W"] == info[team_b]["W"]:

                equal_pairs.append(
                    (
                        team_a,
                        team_b,
                        int(info[team_a]["W"])
                    )
                )

    equal_record_summary.append({
        "Season_End": year,
        "Equal_Record_Pairs": len(equal_pairs)
    })

equal_record_summary = pd.DataFrame(
    equal_record_summary
)

print(
    "Total equal-record playoff-team pairs:",
    equal_record_summary["Equal_Record_Pairs"].sum()
)

print(
    "Average per season:",
    round(
        equal_record_summary["Equal_Record_Pairs"].mean(),
        2
    )
)

display(
    equal_record_summary[
        equal_record_summary["Equal_Record_Pairs"] > 0
    ]
)

Total equal-record playoff-team pairs: 180
Average per season: 5.45


,Season_End,Equal_Record_Pairs
0,1993,3
1,1994,7
2,1995,2
3,1996,2
4,1997,6
5,1998,4
6,1999,11
7,2000,5
8,2001,4
9,2002,7


In [72]:
def build_actual_home_court_lookup(target_end_year):
    """
    For teams that actually met in the historical postseason,
    record which team hosted Game 1.
    """

    games = playoff_games[
        playoff_games["Season_End"] == target_end_year
    ].sort_values("Date")

    lookup = {}
    seen_pairs = set()

    for _, game in games.iterrows():

        team_a = game["Away_Team"]
        team_b = game["Home_Team"]

        pair = frozenset([team_a, team_b])

        if pair not in seen_pairs:

            seen_pairs.add(pair)

            # First game of the series
            lookup[pair] = game["Home_Team"]

    return lookup

In [73]:
def determine_home_court_fast(
    team_a,
    team_b,
    team_info_cache,
    actual_home_court_lookup
):
    """
    Determine home-court advantage.

    Priority:
    1. Better regular-season record
    2. Actual historical home court if these teams met
    3. SRS fallback for hypothetical equal-record matchups
    """

    info_a = team_info_cache[team_a]
    info_b = team_info_cache[team_b]

    # Better record
    if info_a["W"] > info_b["W"]:
        return team_a, team_b

    if info_b["W"] > info_a["W"]:
        return team_b, team_a

    # Equal record: use actual historical assignment if known
    pair = frozenset([team_a, team_b])

    if pair in actual_home_court_lookup:

        historical_home = actual_home_court_lookup[pair]

        if historical_home == team_a:
            return team_a, team_b

        return team_b, team_a

    # Hypothetical matchup with equal records:
    # deterministic documented fallback
    if info_a["SRS"] >= info_b["SRS"]:
        return team_a, team_b

    return team_b, team_a

In [74]:
def simulate_historical_postseason_fast(
    target_end_year,
    historical_bracket,
    probability_cache,
    team_info_cache,
    actual_home_court_lookup,
    rng
):
    """
    Fully optimized historical postseason simulation.

    Uses:
    - reconstructed historical bracket topology
    - historically correct series formats
    - cached game probabilities
    - cached team information
    - actual historical home-court assignments for
      equal-record teams when available
    """

    # -----------------------------
    # ROUND 1
    # -----------------------------

    round1_winners = {}

    for series in historical_bracket["Round1"]:

        series_index = series["Series_Index"]

        team_a = series["Team_1"]
        team_b = series["Team_2"]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache,
                actual_home_court_lookup
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=1,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round1_winners[series_index] = winner

    # -----------------------------
    # ROUND 2
    # -----------------------------

    round2_winners = {}

    for branch in historical_bracket["Round2"]:

        round2_index = branch["Round2_Index"]

        team_a = round1_winners[
            branch["Round1_Series_A"]
        ]

        team_b = round1_winners[
            branch["Round1_Series_B"]
        ]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache,
                actual_home_court_lookup
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=2,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round2_winners[round2_index] = winner

    # -----------------------------
    # CONFERENCE FINALS
    # -----------------------------

    round3_winners = {}

    for branch in historical_bracket["Round3"]:

        round3_index = branch["Round3_Index"]

        team_a = round2_winners[
            branch["Round2_Series_A"]
        ]

        team_b = round2_winners[
            branch["Round2_Series_B"]
        ]

        home_court_team, opponent = (
            determine_home_court_fast(
                team_a,
                team_b,
                team_info_cache,
                actual_home_court_lookup
            )
        )

        winner = simulate_historical_series_fast(
            target_end_year=target_end_year,
            round_number=3,
            home_court_team=home_court_team,
            opponent=opponent,
            probability_cache=probability_cache,
            rng=rng
        )

        round3_winners[round3_index] = winner

    # -----------------------------
    # NBA FINALS
    # -----------------------------

    finals_info = historical_bracket["Finals"]

    team_a = round3_winners[
        finals_info["Round3_Series_A"]
    ]

    team_b = round3_winners[
        finals_info["Round3_Series_B"]
    ]

    home_court_team, opponent = (
        determine_home_court_fast(
            team_a,
            team_b,
            team_info_cache,
            actual_home_court_lookup
        )
    )

    champion = simulate_historical_series_fast(
        target_end_year=target_end_year,
        round_number=4,
        home_court_team=home_court_team,
        opponent=opponent,
        probability_cache=probability_cache,
        rng=rng
    )

    return champion

In [75]:
actual_home_court_2016 = (
    build_actual_home_court_lookup(2016)
)

rng = np.random.default_rng(42)

test_champion = simulate_historical_postseason_fast(
    target_end_year=2016,
    historical_bracket=historical_bracket_2016,
    probability_cache=historical_probability_cache_2016,
    team_info_cache=team_info_cache_2016,
    actual_home_court_lookup=actual_home_court_2016,
    rng=rng
)

print(
    "Test simulated 2016 champion:",
    test_champion
)

Test simulated 2016 champion: Golden State Warriors


In [76]:
# ============================================================
# FULL HISTORICAL PRODUCTION RUN
# 1993-2025 | 10,000 simulations per postseason
# ============================================================

n_simulations = 10000

all_championship_results = []

production_start = time.perf_counter()

for year in range(1993, 2026):

    season_start = time.perf_counter()

    # -----------------------------------
    # Build everything needed for season
    # -----------------------------------

    model = train_probability_model(year)

    bracket = build_historical_bracket(year)

    probability_cache = build_historical_probability_cache(
        target_end_year=year,
        historical_bracket=bracket,
        model=model
    )

    team_info_cache = build_team_info_cache(year)

    actual_home_court_lookup = (
        build_actual_home_court_lookup(year)
    )

    # Get all 16 playoff teams
    playoff_teams = set()

    for series in bracket["Round1"]:
        playoff_teams.add(series["Team_1"])
        playoff_teams.add(series["Team_2"])

    # -----------------------------------
    # Simulate postseason 10,000 times
    # -----------------------------------

    champions = Counter()

    # Different but reproducible seed for each season
    rng = np.random.default_rng(year)

    for _ in range(n_simulations):

        champion = simulate_historical_postseason_fast(
            target_end_year=year,
            historical_bracket=bracket,
            probability_cache=probability_cache,
            team_info_cache=team_info_cache,
            actual_home_court_lookup=actual_home_court_lookup,
            rng=rng
        )

        champions[champion] += 1

    # -----------------------------------
    # Save every playoff team's result
    # -----------------------------------

    for team in playoff_teams:

        championship_count = champions.get(
            team,
            0
        )

        championship_probability = (
            championship_count /
            n_simulations
        )

        all_championship_results.append({
            "Season_End": year,
            "Season":
                f"{year - 1}-{str(year)[-2:]}",
            "Team": team,
            "Simulated_Championships":
                championship_count,
            "Championship_Probability":
                championship_probability
        })

    season_elapsed = (
        time.perf_counter() -
        season_start
    )

    actual_champion = get_series_winner(
        year,
        get_unique_playoff_matchups(year).iloc[14]["Team_1"],
        get_unique_playoff_matchups(year).iloc[14]["Team_2"]
    )

    actual_champion_probability = (
        champions.get(actual_champion, 0) /
        n_simulations
    )

    print(
        f"{year}: "
        f"{season_elapsed:.2f}s | "
        f"Actual champion: {actual_champion} | "
        f"Model probability: "
        f"{actual_champion_probability:.1%}"
    )


# ============================================================
# COMBINE RESULTS
# ============================================================

historical_simulation_results = pd.DataFrame(
    all_championship_results
)

historical_simulation_results = (
    historical_simulation_results
    .sort_values(
        [
            "Season_End",
            "Championship_Probability"
        ],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)


# ============================================================
# SAVE MASTER DATASET
# ============================================================

historical_simulation_results.to_csv(
    "../data/nba_championship_probabilities_1993_2025.csv",
    index=False
)


# ============================================================
# SUMMARY
# ============================================================

production_elapsed = (
    time.perf_counter() -
    production_start
)

print("\n======================================")
print("PRODUCTION RUN COMPLETE")
print("======================================")

print(
    "Seasons simulated:",
    historical_simulation_results[
        "Season_End"
    ].nunique()
)

print(
    "Team-season rows:",
    len(historical_simulation_results)
)

print(
    "Total simulated postseasons:",
    33 * n_simulations
)

print(
    f"Total runtime: "
    f"{production_elapsed:.2f} seconds"
)

print(
    "\nMaster dataset saved to:"
)

print(
    "../data/"
    "nba_championship_probabilities_1993_2025.csv"
)

1993: 0.33s | Actual champion: Chicago Bulls | Model probability: 16.4%
1994: 0.31s | Actual champion: Houston Rockets | Model probability: 4.6%
1995: 0.30s | Actual champion: Houston Rockets | Model probability: 0.2%
1996: 0.30s | Actual champion: Chicago Bulls | Model probability: 66.9%
1997: 0.30s | Actual champion: Chicago Bulls | Model probability: 53.8%
1998: 0.31s | Actual champion: Chicago Bulls | Model probability: 27.9%
1999: 0.32s | Actual champion: San Antonio Spurs | Model probability: 39.7%
2000: 0.31s | Actual champion: Los Angeles Lakers | Model probability: 40.4%
2001: 0.33s | Actual champion: Los Angeles Lakers | Model probability: 3.6%
2002: 0.31s | Actual champion: Los Angeles Lakers | Model probability: 23.6%
2003: 0.33s | Actual champion: San Antonio Spurs | Model probability: 19.7%
2004: 0.33s | Actual champion: Detroit Pistons | Model probability: 12.7%
2005: 0.33s | Actual champion: San Antonio Spurs | Model probability: 41.0%
2006: 0.33s | Actual champion: Mia

In [77]:
# ============================================================
# MASTER SIMULATION INTEGRITY CHECKS
# ============================================================

results = historical_simulation_results.copy()

print("=== BASIC CHECKS ===")

print("Rows:", len(results))
print(
    "Seasons:",
    results["Season_End"].nunique()
)

print(
    "Duplicate team-seasons:",
    results.duplicated(
        subset=["Season_End", "Team"]
    ).sum()
)

print(
    "Missing values:",
    results.isna().sum().sum()
)


# ---------------------------------------
# Championship probabilities by season
# ---------------------------------------

probability_sums = (
    results
    .groupby("Season_End")
    ["Championship_Probability"]
    .sum()
)

print("\n=== PROBABILITY SUMS ===")

print(
    "Minimum:",
    probability_sums.min()
)

print(
    "Maximum:",
    probability_sums.max()
)

print(
    "All approximately 1.0:",
    np.allclose(
        probability_sums.values,
        1.0
    )
)


# ---------------------------------------
# Simulated championships by season
# ---------------------------------------

championship_count_sums = (
    results
    .groupby("Season_End")
    ["Simulated_Championships"]
    .sum()
)

print("\n=== SIMULATION COUNTS ===")

print(
    "Minimum:",
    championship_count_sums.min()
)

print(
    "Maximum:",
    championship_count_sums.max()
)

print(
    "All exactly 10,000:",
    (
        championship_count_sums
        == 10000
    ).all()
)


# ---------------------------------------
# Teams per postseason
# ---------------------------------------

teams_per_season = (
    results
    .groupby("Season_End")
    ["Team"]
    .nunique()
)

print("\n=== PLAYOFF TEAMS PER SEASON ===")

print(teams_per_season.value_counts().sort_index())

print(
    "All seasons have 16 teams:",
    (teams_per_season == 16).all()
)


# ---------------------------------------
# Probability bounds
# ---------------------------------------

print("\n=== PROBABILITY BOUNDS ===")

print(
    "Minimum probability:",
    results[
        "Championship_Probability"
    ].min()
)

print(
    "Maximum probability:",
    results[
        "Championship_Probability"
    ].max()
)

print(
    "All probabilities valid:",
    results[
        "Championship_Probability"
    ].between(0, 1).all()
)

=== BASIC CHECKS ===
Rows: 528
Seasons: 33
Duplicate team-seasons: 0
Missing values: 0

=== PROBABILITY SUMS ===
Minimum: 1.0
Maximum: 1.0
All approximately 1.0: True

=== SIMULATION COUNTS ===
Minimum: 10000
Maximum: 10000
All exactly 10,000: True

=== PLAYOFF TEAMS PER SEASON ===
Team
16    33
Name: count, dtype: int64
All seasons have 16 teams: True

=== PROBABILITY BOUNDS ===
Minimum probability: 0.0
Maximum probability: 0.7535
All probabilities valid: True


In [78]:
# ============================================================
# DEEP DIVE INTO SELECTED SEASONS
# ============================================================

seasons_to_inspect = [1995, 2001, 2016, 2018]

inspection_rows = []

for year in seasons_to_inspect:

    sim = results[
        results["Season_End"] == year
    ].copy()

    regular = team_data[
        team_data["Season_End"] == year
    ][
        ["Team", "W", "L", "SRS", "ORtg", "DRtg", "NRtg"]
    ].copy()

    season_view = sim.merge(
        regular,
        on="Team",
        how="left"
    )

    # Identify actual champion
    series = get_unique_playoff_matchups(year)
    finals = series.iloc[14]

    actual_champion = get_series_winner(
        year,
        finals["Team_1"],
        finals["Team_2"]
    )

    season_view["Actual_Champion"] = (
        season_view["Team"] == actual_champion
    )

    season_view = season_view.sort_values(
        "Championship_Probability",
        ascending=False
    )

    inspection_rows.append(season_view)


inspection_table = pd.concat(
    inspection_rows,
    ignore_index=True
)

display(
    inspection_table[
        [
            "Season",
            "Team",
            "W",
            "L",
            "SRS",
            "Championship_Probability",
            "Actual_Champion"
        ]
    ].style.format({
        "SRS": "{:.2f}",
        "Championship_Probability": "{:.1%}"
    })
)

,Season,Team,W,L,SRS,Championship_Probability,Actual_Champion
0,1994-95,Utah Jazz,60.000000,22.000000,7.76,29.4%,False
1,1994-95,Seattle SuperSonics,57.000000,25.000000,7.91,24.1%,False
2,1994-95,Orlando Magic,57.000000,25.000000,6.44,22.7%,False
3,1994-95,San Antonio Spurs,62.000000,20.000000,5.90,12.0%,False
4,1994-95,Chicago Bulls,47.000000,35.000000,4.32,2.7%,False
5,1994-95,Indiana Pacers,52.000000,30.000000,3.35,2.4%,False
6,1994-95,New York Knicks,55.000000,27.000000,2.78,2.1%,False
7,1994-95,Phoenix Suns,59.000000,23.000000,3.86,1.9%,False
8,1994-95,Charlotte Hornets,50.000000,32.000000,2.87,1.2%,False
9,1994-95,Portland Trail Blazers,44.000000,38.000000,3.80,0.7%,False


In [79]:
# ============================================================
# CHAMPIONSHIP-PROBABILITY CALIBRATION
# ============================================================

championship_validation = results.copy()

# Identify the actual champion for every season
actual_champions = {}

for year in range(1993, 2026):

    series = get_unique_playoff_matchups(year)
    finals = series.iloc[14]

    actual_champion = get_series_winner(
        year,
        finals["Team_1"],
        finals["Team_2"]
    )

    actual_champions[year] = actual_champion


championship_validation["Actual_Champion"] = (
    championship_validation.apply(
        lambda row:
            row["Team"] ==
            actual_champions[row["Season_End"]],
        axis=1
    )
)


# Probability bins
championship_validation["Probability_Bin"] = pd.cut(
    championship_validation[
        "Championship_Probability"
    ],
    bins=[
        0.00,
        0.01,
        0.05,
        0.10,
        0.20,
        0.30,
        0.40,
        0.50,
        0.60,
        0.80,
        1.00
    ],
    include_lowest=True
)


championship_calibration = (
    championship_validation
    .groupby(
        "Probability_Bin",
        observed=True
    )
    .agg(
        Team_Seasons=(
            "Actual_Champion",
            "size"
        ),
        Avg_Predicted_Probability=(
            "Championship_Probability",
            "mean"
        ),
        Actual_Championship_Rate=(
            "Actual_Champion",
            "mean"
        ),
        Actual_Championships=(
            "Actual_Champion",
            "sum"
        )
    )
    .reset_index()
)


championship_calibration["Difference"] = (
    championship_calibration[
        "Actual_Championship_Rate"
    ]
    -
    championship_calibration[
        "Avg_Predicted_Probability"
    ]
)


display(
    championship_calibration.style.format({
        "Avg_Predicted_Probability": "{:.1%}",
        "Actual_Championship_Rate": "{:.1%}",
        "Difference": "{:+.1%}"
    })
)

,Probability_Bin,Team_Seasons,Avg_Predicted_Probability,Actual_Championship_Rate,Actual_Championships,Difference
0,"(-0.001, 0.01]",284,0.3%,0.4%,1,+0.1%
1,"(0.01, 0.05]",105,2.6%,3.8%,4,+1.2%
2,"(0.05, 0.1]",39,7.1%,12.8%,5,+5.8%
3,"(0.1, 0.2]",48,14.1%,14.6%,7,+0.5%
4,"(0.2, 0.3]",18,25.0%,16.7%,3,-8.3%
5,"(0.3, 0.4]",15,35.4%,26.7%,4,-8.7%
6,"(0.4, 0.5]",8,43.6%,25.0%,2,-18.6%
7,"(0.5, 0.6]",4,52.0%,25.0%,1,-27.0%
8,"(0.6, 0.8]",7,66.8%,85.7%,6,+18.9%


In [80]:
# ============================================================
# ACTUAL CHAMPIONS — SIMULATED TITLE PROBABILITIES
# ============================================================

actual_champion_rows = []

for year in range(1993, 2026):

    champion = actual_champions[year]

    champion_result = results[
        (results["Season_End"] == year) &
        (results["Team"] == champion)
    ].iloc[0]

    regular_season = team_data[
        (team_data["Season_End"] == year) &
        (team_data["Team"] == champion)
    ].iloc[0]

    actual_champion_rows.append({
        "Season": champion_result["Season"],
        "Season_End": year,
        "Champion": champion,
        "Wins": int(regular_season["W"]),
        "Losses": int(regular_season["L"]),
        "SRS": regular_season["SRS"],
        "Simulated_Championships":
            champion_result["Simulated_Championships"],
        "Championship_Probability":
            champion_result["Championship_Probability"]
    })


actual_champion_results = pd.DataFrame(
    actual_champion_rows
)

actual_champion_results = (
    actual_champion_results
    .sort_values(
        "Championship_Probability",
        ascending=True
    )
    .reset_index(drop=True)
)

actual_champion_results.index = (
    actual_champion_results.index + 1
)

display(
    actual_champion_results.style.format({
        "SRS": "{:.2f}",
        "Championship_Probability": "{:.1%}"
    })
)

,Season,Season_End,Champion,Wins,Losses,SRS,Simulated_Championships,Championship_Probability
1,1994-95,1995,Houston Rockets,47,35,2.32,25,0.2%
2,2000-01,2001,Los Angeles Lakers,56,26,3.74,362,3.6%
3,2005-06,2006,Miami Heat,52,30,3.59,366,3.7%
4,2010-11,2011,Dallas Mavericks,57,25,4.41,378,3.8%
5,1993-94,1994,Houston Rockets,58,24,4.19,457,4.6%
6,2009-10,2010,Los Angeles Lakers,57,25,4.78,706,7.1%
7,2015-16,2016,Cleveland Cavaliers,57,25,5.45,716,7.2%
8,2022-23,2023,Denver Nuggets,53,29,3.04,875,8.8%
9,2021-22,2022,Golden State Warriors,53,29,5.52,947,9.5%
10,2017-18,2018,Golden State Warriors,58,24,5.79,974,9.7%


In [81]:
actual_champion_results.to_csv(
    "../data/nba_actual_champion_probabilities_1993_2025.csv",
    index=False
)

print("Actual champion probability table saved.")

Actual champion probability table saved.


In [82]:
# ============================================================
# EXPECTED VS ACTUAL CHAMPIONSHIPS
# ============================================================

# Start with every team-season simulation result
expected_titles = (
    results
    .groupby("Team", as_index=False)
    .agg(
        Playoff_Appearances=("Season_End", "count"),
        Expected_Championships=(
            "Championship_Probability",
            "sum"
        )
    )
)

# Count actual championships
actual_title_counts = (
    pd.Series(actual_champions)
    .value_counts()
    .rename_axis("Team")
    .reset_index(name="Actual_Championships")
)

# Merge
franchise_results = expected_titles.merge(
    actual_title_counts,
    on="Team",
    how="left"
)

franchise_results[
    "Actual_Championships"
] = (
    franchise_results[
        "Actual_Championships"
    ]
    .fillna(0)
    .astype(int)
)

# Actual minus expected
franchise_results["Titles_Above_Expected"] = (
    franchise_results["Actual_Championships"] -
    franchise_results["Expected_Championships"]
)

# Sort by expected championships
franchise_results = (
    franchise_results
    .sort_values(
        "Expected_Championships",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    franchise_results.style.format({
        "Expected_Championships": "{:.2f}",
        "Titles_Above_Expected": "{:+.2f}"
    })
)

,Team,Playoff_Appearances,Expected_Championships,Actual_Championships,Titles_Above_Expected
0,San Antonio Spurs,26,4.42,5,+0.58
1,Boston Celtics,23,2.56,2,-0.56
2,Chicago Bulls,18,2.39,4,+1.61
3,Golden State Warriors,12,2.25,4,+1.75
4,Utah Jazz,22,1.80,0,-1.80
5,Los Angeles Lakers,24,1.66,6,+4.34
6,Oklahoma City Thunder,12,1.57,1,-0.57
7,Phoenix Suns,19,1.48,0,-1.48
8,Miami Heat,25,1.46,3,+1.54
9,Cleveland Cavaliers,17,1.46,1,-0.46


In [83]:
# ============================================================
# COMBINE HISTORICAL TEAM NAMES INTO FRANCHISES
# ============================================================

franchise_name_map = {
    "Seattle SuperSonics": "Oklahoma City Thunder / Seattle SuperSonics",
    "Oklahoma City Thunder": "Oklahoma City Thunder / Seattle SuperSonics",

    "New Jersey Nets": "Brooklyn Nets / New Jersey Nets",
    "Brooklyn Nets": "Brooklyn Nets / New Jersey Nets",

    "Charlotte Bobcats": "Charlotte Hornets",
    "Charlotte Hornets": "Charlotte Hornets",

    "Washington Bullets": "Washington Wizards / Bullets",
    "Washington Wizards": "Washington Wizards / Bullets",

    "New Orleans Hornets": "New Orleans Pelicans / Hornets",
    "New Orleans Pelicans": "New Orleans Pelicans / Hornets"
}

franchise_level = results.copy()

franchise_level["Franchise"] = (
    franchise_level["Team"]
    .replace(franchise_name_map)
)

# Expected titles
franchise_expected = (
    franchise_level
    .groupby("Franchise", as_index=False)
    .agg(
        Playoff_Appearances=("Season_End", "count"),
        Expected_Championships=(
            "Championship_Probability",
            "sum"
        )
    )
)

# Map actual champions to franchise names
actual_champion_franchises = pd.Series(
    {
        year: franchise_name_map.get(team, team)
        for year, team in actual_champions.items()
    }
)

franchise_actual = (
    actual_champion_franchises
    .value_counts()
    .rename_axis("Franchise")
    .reset_index(name="Actual_Championships")
)

# Merge
franchise_summary = franchise_expected.merge(
    franchise_actual,
    on="Franchise",
    how="left"
)

franchise_summary["Actual_Championships"] = (
    franchise_summary["Actual_Championships"]
    .fillna(0)
    .astype(int)
)

franchise_summary["Titles_Above_Expected"] = (
    franchise_summary["Actual_Championships"]
    -
    franchise_summary["Expected_Championships"]
)

franchise_summary = (
    franchise_summary
    .sort_values(
        "Titles_Above_Expected",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    franchise_summary.style.format({
        "Expected_Championships": "{:.2f}",
        "Titles_Above_Expected": "{:+.2f}"
    })
)

,Franchise,Playoff_Appearances,Expected_Championships,Actual_Championships,Titles_Above_Expected
0,Los Angeles Lakers,24,1.66,6,+4.34
1,Golden State Warriors,12,2.25,4,+1.75
2,Chicago Bulls,18,2.39,4,+1.61
3,Miami Heat,25,1.46,3,+1.54
4,Houston Rockets,21,0.72,2,+1.28
5,Denver Nuggets,19,0.32,1,+0.68
6,San Antonio Spurs,26,4.42,5,+0.58
7,Toronto Raptors,13,0.70,1,+0.30
8,Detroit Pistons,15,0.83,1,+0.17
9,Washington Wizards / Bullets,10,0.01,0,-0.01


In [84]:
# ============================================================
# SEASON-BY-SEASON TITLE OPPORTUNITIES
# ============================================================

teams_to_inspect = [
    "Los Angeles Lakers",
    "Golden State Warriors",
    "Chicago Bulls",
    "Miami Heat",
    "Houston Rockets",
    "San Antonio Spurs",
    "Utah Jazz",
    "Phoenix Suns",
    "Seattle SuperSonics",
    "Oklahoma City Thunder"
]

opportunity_table = results[
    results["Team"].isin(teams_to_inspect)
].copy()

# Mark actual champions
opportunity_table["Actual_Champion"] = (
    opportunity_table.apply(
        lambda row:
            row["Team"] ==
            actual_champions[row["Season_End"]],
        axis=1
    )
)

opportunity_table = (
    opportunity_table
    .sort_values(
        ["Team", "Season_End"]
    )
    .reset_index(drop=True)
)

display(
    opportunity_table[
        [
            "Season",
            "Team",
            "Championship_Probability",
            "Actual_Champion"
        ]
    ].style.format({
        "Championship_Probability": "{:.1%}"
    })
)

,Season,Team,Championship_Probability,Actual_Champion
0,1992-93,Chicago Bulls,16.4%,True
1,1993-94,Chicago Bulls,1.1%,False
2,1994-95,Chicago Bulls,2.7%,False
3,1995-96,Chicago Bulls,66.9%,True
4,1996-97,Chicago Bulls,53.8%,True
5,1997-98,Chicago Bulls,27.9%,True
6,2004-05,Chicago Bulls,0.1%,False
7,2005-06,Chicago Bulls,0.1%,False
8,2006-07,Chicago Bulls,7.2%,False
9,2008-09,Chicago Bulls,0.0%,False


In [85]:
# Top five championship opportunities for each selected team
top_opportunities = (
    opportunity_table
    .sort_values(
        "Championship_Probability",
        ascending=False
    )
    .groupby("Team")
    .head(5)
    .sort_values(
        ["Team", "Championship_Probability"],
        ascending=[True, False]
    )
)

display(
    top_opportunities[
        [
            "Season",
            "Team",
            "Championship_Probability",
            "Actual_Champion"
        ]
    ].style.format({
        "Championship_Probability": "{:.1%}"
    })
)

,Season,Team,Championship_Probability,Actual_Champion
3,1995-96,Chicago Bulls,66.9%,True
4,1996-97,Chicago Bulls,53.8%,True
12,2011-12,Chicago Bulls,33.2%,False
5,1997-98,Chicago Bulls,27.9%,True
11,2010-11,Chicago Bulls,27.2%,False
24,2016-17,Golden State Warriors,75.3%,True
22,2014-15,Golden State Warriors,60.3%,True
23,2015-16,Golden State Warriors,48.6%,False
26,2018-19,Golden State Warriors,17.4%,False
25,2017-18,Golden State Warriors,9.7%,True


In [86]:
# ============================================================
# SAVE FRANCHISE-LEVEL RESULTS
# ============================================================

franchise_summary.to_csv(
    "../data/nba_franchise_expected_vs_actual_1993_2025.csv",
    index=False
)

print("Franchise results saved.")
print("Rows:", len(franchise_summary))

Franchise results saved.
Rows: 30
